# IQL on HalfCheetah-v5 — notebook outline
**Status: M0 scaffold, unexecuted.** This is a handoff to Person 5, not the final course notebook.

Use the repository's Python 3.11 locked environment. Install the package from a recorded commit before running cells. Colab runtime compatibility and the final Run all are future team acceptance gates.

See `PROJECT_PLAN.md`, `CONTRIBUTING.md` and `docs/INTERFACES.md`.


## 1. Introduction
Person 5: explain offline RL, project goal and course deliverables.


In [ ]:
from iql_project import __version__
from iql_project.cli import main

print("Package version:", __version__)
assert main(["check"]) == 0

## 2. Environment
HalfCheetah-v5 observes 17 continuous positions/velocities and controls six joint torques in [-1, 1]. Its goal is forward motion with low control effort; default reward is horizontal velocity minus 0.1 times the squared action norm. The dataset records a 1,000-step horizon. Time-limit truncations end episodes while preserving the Q bootstrap.

Person 2 implements `make_evaluation_env(config, render_mode=None)` using the dataset collection EnvSpec. Recovery and ten finite random-action steps passed. The locked environment now uses MuJoCo 3.2.3, matching the dataset collection requirement. The original Person 2 verification used 3.3.7; see [simulator compatibility](../docs/SIMULATOR_COMPATIBILITY.md) for the version decision and [the Person 2 handoff](../docs/ENVIRONMENT_DATASET.md) for exact commands and settings.


## 3. Offline Dataset
`mujoco/halfcheetah/medium-v0` contains 1,000 episodes / 1,000,000 transitions. Person 2 validated the complete dataset: untransformed reward mean/std = 12.08921 / 4.69655; episode return mean/std = 12,089.21036 / 3,008.94886. Every episode has 1,000 steps; there are zero true terminations and 1,000 truncations. These are offline dataset statistics, not trained-policy results.

The loader pairs T actions with observations `[:-1]` and `[1:]` inside each episode only. Numeric batches are float32 with separate bool termination/truncation flags. Mean/population std are fitted only on current offline observations using float64 accumulation; the stored float32 std is floored at 1e-3. Evaluation must reuse `dataset.info` values. Actions and rewards stay in environment units. Source author is Kallinteris Andreas (SB3/TQC); no dataset license is declared in metadata.

Download explicitly before running the example: `python scripts/check_dataset.py --download` from the repository root. If using a custom cache, set `MINARI_DATASETS_PATH` in this notebook process too. The loader itself defaults to `download=False`; see [the Person 2 handoff](../docs/ENVIRONMENT_DATASET.md).


In [ ]:
import numpy as np

from iql_project.config import ProjectConfig
from iql_project.dataset import load_offline_dataset

config = ProjectConfig()
dataset = load_offline_dataset(config)  # local data only; no implicit download
rng = np.random.default_rng(config.seed)
batch = dataset.sample(config.batch_size, rng)
print(len(dataset), batch.observations.shape, batch.actions.shape)
print(dataset.metadata["statistics"])
# Person 3 saves dataset.info with the checkpoint; Person 4 reuses it.
# Optional: dataset.save_metadata("results/person-2/preprocessing.json")

## 4. What is IQL?
Person 5: explain the paper and why dataset actions are used for value learning.


## 5. IQL Architecture
Person 3: expose package network definitions, losses, target updates and policy distribution.


## 6. Manual IQL Update Example
Person 5 with Person 3 review: work one numerical transition through expectile V loss, terminated-masked Q target and capped advantage policy weight. State inverse-temperature convention.


## 7. Implementation
Person 3: import learner from iql_project; demonstrate a finite update after Person 2's loader is merged.


## 8. Training
Persons 3–4: link config, run manifests, training seeds, checkpoint/resume and measured hardware budget.


## 9. Results
Person 4: plot real metrics, compare random baseline and report within/across-seed statistics.


## 10. Evaluation
Person 4: show checkpoint preprocessing, fixed evaluation seeds and raw returns.


## 11. Demo and References
Persons 4–5: embed playable MP4 and cite paper/data/adapted code. Person 1 verifies final Run all in clean Colab after all role deliverables arrive.
